In [5]:
!uv pip install "compel==2.2.1.dev5"

Using Python 3.12.8 environment at: /Users/damian/2.current/InvokeAI/.venv
Resolved 114 packages in 186ms                                       
Prepared 1 package in 113ms                                              
Uninstalled 1 package in 0.85ms
Installed 1 package in 1ms                                  
 - compel==2.2.1
 + compel==2.2.1.dev5


In [1]:
%load_ext autoreload
%autoreload 2

In [40]:
import math
from typing import Tuple, Self
import torch
from collections import defaultdict
from contextlib import contextmanager
from functools import partial

@contextmanager
def collect_attention_maps(unet, text_encoder_hidden_size: int):
    collector = CrossAttentionMapCollector(unet, text_encoder_hidden_size)
    original_sdp_func = torch.nn.functional.scaled_dot_product_attention
    try:
        yield collector
    finally:
        collector.remove_hooks()
        # in case something went wrong - forward post-hooks might not have been called, so make sure we un-monkeypatch SDP
        torch.nn.functional.scaled_dot_product_attention = original_sdp_func


class CrossAttentionMapCollector:
    def __init__(self, unet, text_encoder_hidden_size: int, verbose: bool=True) -> None:
        self.unet = unet
        self.text_encoder_hidden_size = text_encoder_hidden_size
        self.attention_maps = defaultdict(list)
        self.hooks = []
        self._original_sdp_func = None
        self.verbose = verbose
        self.register_hooks()

    def register_hooks(self):
        """Register forward hooks on all cross-attention modules in the UNet."""

        def _sdp_with_map_saving(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None, target: CrossAttentionMapCollector=None, map_name: str=""):
            attn_output, attn_weights = _scaled_dot_product_attention_with_weight_return(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
            if self.verbose and map_name not in target.attention_maps:
                # log on first addition
                print(f"storing maps of size {attn_weights.shape} for '{map_name}'")
            target.attention_maps[map_name].append(attn_weights.detach().cpu())
            return attn_output

        def pre_hook_fn(module, input, name):
            # oerride sdp
            self._original_sdp_func = torch.nn.functional.scaled_dot_product_attention
            torch.nn.functional.scaled_dot_product_attention = partial(_sdp_with_map_saving, target=self, map_name=name)

        def post_hook_fn(module, input, output):
            # restore sdp
            torch.nn.functional.scaled_dot_product_attention = self._original_sdp_func

        # Find all cross-attention modules and register hooks
        # attn_modules = {n: m for n, m in pipeline.unet.named_modules() if 'attn' in n.lower() and hasattr(m, 'to_q') and hasattr(m, 'to_k') and hasattr(m, 'to_v') and m.cross_attention_dim == pipeline.text_encoder.config.hidden_size}
        for name, module in self.unet.named_modules():
            # Look for cross-attention modules
            if 'attn' in name.lower() and (
                hasattr(module, "to_q") and hasattr(module, "to_k") and hasattr(module, "to_v")
            ) and (
                module.cross_attention_dim == self.text_encoder_hidden_size
            ):
                module: torch.nn.Module
                pre_hook = module.register_forward_pre_hook(
                    lambda mod, inp, n=name: pre_hook_fn(mod, inp, n)
                )
                self.hooks.append(pre_hook)
                post_hook = module.register_forward_hook(post_hook_fn)
                self.hooks.append(post_hook)

    def remove_hooks(self):
        """Remove all registered hooks."""
        for hook in self.hooks:
            hook.remove()
        self.hooks = []

    def clear_maps(self):
        """Clear collected attention maps."""
        self.attention_maps = defaultdict(list)

    def get_attention_maps(self):
        """Return the collected cross-attention maps."""
        return self.attention_maps

def _scaled_dot_product_attention_with_weight_return(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None) -> Tuple[torch.Tensor, torch.Tensor]:
    # Efficient implementation equivalent to the following:
    L, S = query.size(-2), key.size(-2)
    scale_factor = 1 / math.sqrt(query.size(-1)) if scale is None else scale
    attn_bias = torch.zeros(L, S, dtype=query.dtype)
    if is_causal:
        assert attn_mask is None
        temp_mask = torch.ones(L, S, dtype=torch.bool).tril(diagonal=0)
        attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf"))
        attn_bias.to(query.dtype)

    if attn_mask is not None:
        if attn_mask.dtype == torch.bool:
            attn_mask.masked_fill_(attn_mask.logical_not(), float("-inf"))
        else:
            attn_bias += attn_mask
    attn_weight = query @ key.transpose(-2, -1) * scale_factor
    attn_weight += attn_bias.to(attn_weight.device)
    attn_weight = torch.softmax(attn_weight, dim=-1)
    attn_weight_with_dropout = torch.dropout(attn_weight, dropout_p, train=True)

    return attn_weight_with_dropout @ value, attn_weight


In [41]:
from diffusers import StableDiffusionPipeline
pipeline = StableDiffusionPipeline.from_pretrained('runwayml/stable-diffusion-v1-5', torch_dtype=torch.bfloat16).to('mps')

Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00,  7.76it/s]


In [80]:
from compel import CompelForSD
compel = CompelForSD(pipeline)
conditioning = compel("A photo of an astronaut riding a horse on mars", negative_prompt="ugly, blurry")


In [81]:
attention_maps = None
with (
    torch.no_grad(),
    collect_attention_maps(pipeline.unet,
                           text_encoder_hidden_size=pipeline.text_encoder.config.hidden_size)
    as collector
):
    image = pipeline(prompt_embeds=conditioning.embeds, negative_prompt_embeds=conditioning.negative_embeds, num_inference_steps=5).images[0]
    attention_maps = collector.get_attention_maps()




  0%|          | 0/5 [00:00<?, ?it/s]

storing map of size torch.Size([2, 8, 4096, 77]) for down_blocks.0.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for down_blocks.0.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 1024, 77]) for down_blocks.1.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 1024, 77]) for down_blocks.1.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 256, 77]) for down_blocks.2.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 256, 77]) for down_blocks.2.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 64, 77]) for mid_block.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 256, 77]) for up_blocks.1.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 256, 77]) for up_blocks.1.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 256, 77]) for

 20%|██        | 1/5 [00:00<00:03,  1.16it/s]

storing map of size torch.Size([2, 8, 1024, 77]) for up_blocks.2.attentions.2.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for up_blocks.3.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for up_blocks.3.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for up_blocks.3.attentions.2.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for down_blocks.0.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for down_blocks.0.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 1024, 77]) for down_blocks.1.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 1024, 77]) for down_blocks.1.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 256, 77]) for down_blocks.2.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 256, 77]

 40%|████      | 2/5 [00:01<00:01,  1.77it/s]

...

storing map of size torch.Size([2, 8, 256, 77]) for up_blocks.1.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 256, 77]) for up_blocks.1.attentions.2.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 1024, 77]) for up_blocks.2.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 1024, 77]) for up_blocks.2.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 1024, 77]) for up_blocks.2.attentions.2.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for up_blocks.3.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for up_blocks.3.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for up_blocks.3.attentions.2.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for down_blocks.0.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for do

 80%|████████  | 4/5 [00:01<00:00,  2.37it/s]

storing map of size torch.Size([2, 8, 1024, 77]) for up_blocks.2.attentions.2.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for up_blocks.3.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for up_blocks.3.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for up_blocks.3.attentions.2.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for down_blocks.0.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for down_blocks.0.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 1024, 77]) for down_blocks.1.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 1024, 77]) for down_blocks.1.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 256, 77]) for down_blocks.2.attentions.0.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 256, 77]

100%|██████████| 5/5 [00:02<00:00,  2.18it/s]

storing map of size torch.Size([2, 8, 4096, 77]) for up_blocks.3.attentions.1.transformer_blocks.0.attn2
storing map of size torch.Size([2, 8, 4096, 77]) for up_blocks.3.attentions.2.transformer_blocks.0.attn2


In [149]:
import PIL.Image
from torchvision.transforms.functional import resize as torchvision_resize, InterpolationMode

def get_stacked_maps(maps_dict: dict[str, list[torch.Tensor]], latents_width: int, latents_height: int,
                     prompt_index: int, eos_token_index: int=None, drop_bos_eos=True,
                     merge_timesteps = False) -> torch.Tensor:
    """
    Scale all collected attention maps to the same size, blend them together and return as an image.
    latents_width and latents_height are the width and height of the latent space, e.g. 64x64 for 512x512 images with 8x downsampling.
    :return: An image containing a vertical stack of blended attention maps, one for each requested token.
    """
    merged = None
    for key, maps in maps_dict.items():
        maps = torch.stack(maps, dim=0) # [steps, B, heads, (H*W), N]
        maps = torch.swapdims(maps, 0, 1) # [B, steps, heads, (H*W), N]
        assert len(maps.shape) == 5 # [B, steps, heads, (H*W), N]

        maps = maps[prompt_index:prompt_index+1, ...] # [1, steps, heads, (H*W), N]

        # drop padding tokens, bos, eos
        if eos_token_index is not None:
            maps = maps[..., :eos_token_index+1] # [B, steps, heads, (H*W), N]
            if drop_bos_eos:
                maps = maps[..., 1:-1] # [B, steps, heads, (H*W), N]

        # merge all heads together by averaging
        maps = torch.mean(maps, dim=2) # now [B, steps, (H*W), N]
        #maps = torch.mean(maps, dim=1, keepdim=True)

        # maps has shape [B, steps, (H*W), N] for N tokens
        # but we want [B, steps, N, H, W] for torchvision_resize
        this_scale_factor = math.sqrt(maps.shape[2] / (latents_width * latents_height))
        this_maps_height = int(float(latents_height) * this_scale_factor)
        this_maps_width = int(float(latents_width) * this_scale_factor)
        # and we need to do some dimension juggling
        bsz = maps.shape[0]
        num_steps = maps.shape[1]
        num_tokens = maps.shape[-1]
        maps = torch.reshape(torch.swapdims(maps, -2, -1), [bsz, num_steps, num_tokens, this_maps_height, this_maps_width])

        # scale to output size if necessary
        if this_scale_factor != 1:
            # torchvision resize expects [..., H, W]
            maps = maps.reshape(bsz*num_steps, num_tokens, this_maps_height, this_maps_width)
            maps = torchvision_resize(maps, [latents_height, latents_width], InterpolationMode.BICUBIC)
            maps = maps.reshape(bsz, num_steps, num_tokens, latents_height, latents_width)

        # normalize
        maps_min = torch.amin(maps, dim=(-3, -2, -1), keepdim=True)
        maps_range = torch.amax(maps, dim=(-3, -2, -1), keepdim=True) - maps_min
        #print(f"map {key} size {[this_maps_width, this_maps_height]} range {[maps_min, maps_min + maps_range]}")
        maps_normalized = (maps - maps_min) / maps_range
        # expand to (-0.1, 1.1) and clamp
        maps_normalized_expanded = maps_normalized * 1.1 - 0.05
        maps_normalized_expanded_clamped = torch.clamp(maps_normalized_expanded, 0, 1)
        #maps_normalized_expanded_clamped = maps

        # stack tokens vertically
        maps_stacked = torch.reshape(maps_normalized_expanded_clamped, [bsz, num_steps, num_tokens * latents_height, latents_width])
        # map_stacked is [B, steps, (H*W), N]
        if merge_timesteps:
            maps_stacked = torch.mean(maps_stacked, dim=1, keepdim=False)
        else:
            # stack steps horizontally, producing a grid
            maps_stacked = maps_stacked.permute(0, 2, 1, 3).reshape(maps_stacked.shape[0], maps_stacked.shape[2], -1)
        # maps_stacked is now [B, (N*H), (W*steps)] where steps==1 if merge_timesteps is True

        if merged is None:
            merged = maps_stacked
        else:
          # screen blend
            merged = 1 - (1 - maps_stacked)*(1 - merged)

    return merged.squeeze(0)

def get_stacked_maps_images(maps: dict[str, list[torch.Tensor]], latents_width: int, latents_height: int, prompt_index: int, eos_token_index: int=None, merge_timesteps=False) -> PIL.Image:
    merged = get_stacked_maps(maps, latents_width=latents_width, latents_height=latents_height, prompt_index=prompt_index, eos_token_index=eos_token_index, merge_timesteps=merge_timesteps)
    # [(N*H), (W*steps)]
    assert len(merged.shape) == 2
    merged_bytes = merged.mul(0xff).byte()
    return PIL.Image.fromarray(merged_bytes.numpy(), mode='L')


In [150]:
eos_indices = [torch.where(conditioning.tokenization_info['all'][i][0] == 49407)[1][0]
               for i in range(len(conditioning.tokenization_info['all']))]


In [151]:

eos_indices

[tensor(11, device='mps:0'), tensor(4, device='mps:0')]

In [152]:
for prompt_index, eos_index in enumerate(reversed(eos_indices)):
    img = get_stacked_maps_images(attention_maps, latents_width=64, latents_height=64, prompt_index=prompt_index, eos_token_index=eos_index, merge_timesteps=True)
    img.save(f"attention_maps_prompt{prompt_index}.jpg", quality=95)


/var/folders/1s/tbn_rwks3pgb2sx3rd3pct640000gn/T/ipykernel_48209/2658846600.py:81: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return PIL.Image.fromarray(merged_bytes.numpy(), mode='L')


In [19]:

attn_modules = {n: m for n, m in pipeline.unet.named_modules() if 'attn' in n.lower() and hasattr(m, 'to_q') and hasattr(m, 'to_k') and hasattr(m, 'to_v') and m.cross_attention_dim == pipeline.text_encoder.config.hidden_size}

In [20]:
pipeline.text_encoder.config.hidden_size


768

In [148]:
[(n, p.shape) for n, p in pipeline.unet.named_parameters()]


[('conv_in.weight', torch.Size([320, 4, 3, 3])),
 ('conv_in.bias', torch.Size([320])),
 ('time_embedding.linear_1.weight', torch.Size([1280, 320])),
 ('time_embedding.linear_1.bias', torch.Size([1280])),
 ('time_embedding.linear_2.weight', torch.Size([1280, 1280])),
 ('time_embedding.linear_2.bias', torch.Size([1280])),
 ('down_blocks.0.attentions.0.norm.weight', torch.Size([320])),
 ('down_blocks.0.attentions.0.norm.bias', torch.Size([320])),
 ('down_blocks.0.attentions.0.proj_in.weight', torch.Size([320, 320, 1, 1])),
 ('down_blocks.0.attentions.0.proj_in.bias', torch.Size([320])),
 ('down_blocks.0.attentions.0.transformer_blocks.0.norm1.weight',
  torch.Size([320])),
 ('down_blocks.0.attentions.0.transformer_blocks.0.norm1.bias',
  torch.Size([320])),
 ('down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q.weight',
  torch.Size([320, 320])),
 ('down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_k.weight',
  torch.Size([320, 320])),
 ('down_blocks.0.attentions.0.transform

In [15]:
[(type(m), m.cross_attention_dim, m.query_dim) for m in attn_modules.values()]

[(diffusers.models.attention_processor.Attention, 320, 320),
 (diffusers.models.attention_processor.Attention, 768, 320),
 (diffusers.models.attention_processor.Attention, 320, 320),
 (diffusers.models.attention_processor.Attention, 768, 320),
 (diffusers.models.attention_processor.Attention, 640, 640),
 (diffusers.models.attention_processor.Attention, 768, 640),
 (diffusers.models.attention_processor.Attention, 640, 640),
 (diffusers.models.attention_processor.Attention, 768, 640),
 (diffusers.models.attention_processor.Attention, 1280, 1280),
 (diffusers.models.attention_processor.Attention, 768, 1280),
 (diffusers.models.attention_processor.Attention, 1280, 1280),
 (diffusers.models.attention_processor.Attention, 768, 1280),
 (diffusers.models.attention_processor.Attention, 1280, 1280),
 (diffusers.models.attention_processor.Attention, 768, 1280),
 (diffusers.models.attention_processor.Attention, 1280, 1280),
 (diffusers.models.attention_processor.Attention, 768, 1280),
 (diffusers.

In [ ]:
from transformers import CLIPTokenizer

tokenizer: CLIPTokenizer
tokenizer.convert_ids_to_tokens()

In [154]:
pipeline.tokenizer.tokenize("Super cute tiger cub, national geographic award-winning photograph")


['super</w>',
 'cute</w>',
 'tiger</w>',
 'cub</w>',
 ',</w>',
 'national</w>',
 'geographic</w>',
 'award</w>',
 '-</w>',
 'winning</w>',
 'photograph</w>']